# ShanghaiTech Video Anomaly Detection: Hiera-L Feature Extraction Pipeline

This notebook implements a complete, robust, and end-to-end pipeline to extract spatiotemporal features from the **ShanghaiTech** dataset using **Hiera-L (Large)**.

### Pipeline Overview
1. **Google Drive Integration**: Mounts Drive and sets up structured path constants.
2. **Environment & Pinned Dependencies**: Installs `timm`, Hiera (at pinned commit), `decord`, and other necessary libraries.
3. **Dataset Scanning**: Discovers and validates `.avi` files (training) and JPG-frame folders (testing).
4. **Data Loader & Processing Core**: Custom spatiotemporal strided temporal sampling (stride 4, 16 frames per clip, centered alignment) and normalized Hiera preprocessing.
5. **Model Setup**: Loads PyTorch Hub Kinetics-400 fine-tuned Hiera-L model and replaces classifier with an Identity head.
6. **Smoke Testing**: Validates feature extraction correctness on single train/test items.
7. **Atomic, Resumable Feature Extraction**: Safe execution with resume capabilities and temporary-write files.
8. **Feature Statistics Computation**: Running batch calculation of training set mean and standard deviation for downstream standardization.
9. **MULDE-Ready Index Generation**: Generates standard manifests and consolidated configuration/stats files.

*References*:
- **MULDE Paper**: [Micorek et al., CVPR 2024](https://ar5iv.labs.arxiv.org/html/2403.14497v1)
- **Hiera Repository**: [facebookresearch/hiera](https://github.com/facebookresearch/hiera)

## Step 1: Mount Google Drive & Configure Paths

We mount Google Drive to `/content/drive` and configure our workspace paths under `/content/drive/MyDrive/MULDE_ShanghaiTech`. We also define hyperparameters (e.g. random seed, model identifier, and extraction settings).

In [1]:
from sympy import content
import os
import random
import numpy as np
import torch

# 1. Mount Google Drive
try:
    from google.colab import drive
    print("Colab environment detected. Mounting Google Drive...")
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Google Colab. Skipping Google Drive mount.")

# 2. Define Workspace Paths
# The path constants can be edited at the top of the notebook:
DRIVE_ROOT = "/content/drive/MyDrive/MULDE"

# Raw inputs
TRAIN_VIDEO_DIR = os.path.join("/content/", "shanghaitech/training/videos")
TEST_FRAME_DIR = os.path.join("/content/", "shanghaitech/testing/frames")

# Extracted outputs
FEATURE_DIR = os.path.join(DRIVE_ROOT, "features/hiera_large_16x224_mae_k400_ft_k400_centered_s4")
TRAIN_FEATURE_DIR = os.path.join(FEATURE_DIR, "train")
TEST_FEATURE_DIR = os.path.join(FEATURE_DIR, "test")

# Indexing files
MANIFEST_TRAIN = os.path.join(FEATURE_DIR, "manifest_train.csv")
MANIFEST_TEST = os.path.join(FEATURE_DIR, "manifest_test.csv")
FEATURE_STATS_PATH = os.path.join(FEATURE_DIR, "train_feature_stats.npz")
CONFIG_PATH = os.path.join(FEATURE_DIR, "extraction_config.json")

# Create output directories if they don't exist
os.makedirs(TRAIN_FEATURE_DIR, exist_ok=True)
os.makedirs(TEST_FEATURE_DIR, exist_ok=True)

# 3. Model & Feature Configuration Constants
HIERA_COMMIT = "b12b842542ee5c757fcfec8c41f6b56fcbe89b65"
MODEL_NAME = "hiera_large_16x224"
CHECKPOINT_NAME = "mae_k400_ft_k400"
NUM_FRAMES_PER_CLIP = 16
TEMPORAL_STRIDE = 4
IMAGE_SIZE = 224

# 4. Reproducibility & Random Seeds
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    print(f"Random seed set to: {seed}")

seed_everything(42)

print("\n--- Workspace Paths Configuration ---")
print(f"Raw Train Videos: {TRAIN_VIDEO_DIR}")
print(f"Raw Test Frames: {TEST_FRAME_DIR}")
print(f"Feature Storage: {FEATURE_DIR}")

Colab environment detected. Mounting Google Drive...
Mounted at /content/drive
Random seed set to: 42

--- Workspace Paths Configuration ---
Raw Train Videos: /content/shanghaitech/training/videos
Raw Test Frames: /content/shanghaitech/testing/frames
Feature Storage: /content/drive/MyDrive/MULDE/features/hiera_large_16x224_mae_k400_ft_k400_centered_s4


In [2]:
!tar -xzvf "/content/drive/MyDrive/shanghaitech.tar.gz" -C "/content/"

Streaming output truncated to the last 5000 lines.
shanghaitech/testing/frames/08_0077/079.jpg
shanghaitech/testing/frames/08_0077/269.jpg
shanghaitech/testing/frames/08_0077/113.jpg
shanghaitech/testing/frames/08_0077/149.jpg
shanghaitech/testing/frames/08_0077/084.jpg
shanghaitech/testing/frames/08_0077/338.jpg
shanghaitech/testing/frames/08_0077/028.jpg
shanghaitech/testing/frames/08_0077/402.jpg
shanghaitech/testing/frames/08_0077/169.jpg
shanghaitech/testing/frames/08_0077/288.jpg
shanghaitech/testing/frames/08_0077/222.jpg
shanghaitech/testing/frames/08_0077/283.jpg
shanghaitech/testing/frames/08_0077/126.jpg
shanghaitech/testing/frames/08_0077/122.jpg
shanghaitech/testing/frames/08_0077/455.jpg
shanghaitech/testing/frames/08_0077/378.jpg
shanghaitech/testing/frames/08_0077/270.jpg
shanghaitech/testing/frames/08_0077/177.jpg
shanghaitech/testing/frames/08_0077/305.jpg
shanghaitech/testing/frames/08_0077/210.jpg
shanghaitech/testing/frames/08_0077/428.jpg
shanghaitech/testing/fram

## Step 2: Install Pinned Dependencies

We install required external libraries including a PyTorch-compatible `timm`, the official Hiera implementation pinned to the exact recommended commit, `decord` for fast hardware-accelerated video decoding, `opencv-python-headless`, `pandas`, and `tqdm`.

In [3]:
# Install pinned dependencies
# Using -q flag to keep the output clean and professional
print("Installing pip dependencies. This might take a minute...")
!pip install -q timm decord opencv-python-headless pandas tqdm
!pip install -q git+https://github.com/facebookresearch/hiera.git@b12b842542ee5c757fcfec8c41f6b56fcbe89b65

print("\nAll dependencies installed successfully:")
import timm
import hiera
import decord
import cv2
import pandas as pd
import tqdm
print(f" - timm: {timm.__version__}")
print(f" - hiera: installed from commit {HIERA_COMMIT[:8]}")
print(f" - decord: {decord.__version__}")
print(f" - cv2 (OpenCV): {cv2.__version__}")
print(f" - pandas: {pd.__version__}")
print(f" - tqdm: {tqdm.__version__}")

Installing pip dependencies. This might take a minute...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 110.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done

All dependencies installed successfully:


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


 - timm: 1.0.27
 - hiera: installed from commit b12b8425
 - decord: 0.6.0
 - cv2 (OpenCV): 4.13.0
 - pandas: 2.2.2
 - tqdm: 4.67.3


## Step 3: Scan Dataset & Validate Structure

We scan the dataset directory structure to ensure it matches the expected ShanghaiTech layout. We assert that:
- Training raw files are `.avi` videos.
- Testing raw folders contain `.jpg` image frame sequences.
This step checks data integrity and outputs video/folder counts.

In [4]:
import glob

def scan_shanghaitech_dataset():
    print("Scanning dataset directories...")

    # 1. Check if directories exist
    if not os.path.exists(TRAIN_VIDEO_DIR):
        print(f"Warning: Training directory not found: {TRAIN_VIDEO_DIR}")
        train_videos = []
    else:
        train_videos = sorted(glob.glob(os.path.join(TRAIN_VIDEO_DIR, "*.avi")))

    if not os.path.exists(TEST_FRAME_DIR):
        print(f"Warning: Testing directory not found: {TEST_FRAME_DIR}")
        test_folders = []
    else:
        test_folders = sorted([
            os.path.join(TEST_FRAME_DIR, d)
            for d in os.listdir(TEST_FRAME_DIR)
            if os.path.isdir(os.path.join(TEST_FRAME_DIR, d))
        ])

    print(f"Discovered training videos (.avi): {len(train_videos)}")
    print(f"Discovered testing video frame folders: {len(test_folders)}")

    # 2. Perform validation assertions if any files were found
    if len(train_videos) > 0:
        for path in train_videos:
            assert path.lower().endswith(".avi"), f"Validation Error: Train path {path} must be a .avi file."
        print("✓ All training video paths end with .avi")

    if len(test_folders) > 0:
        for folder in test_folders:
            jpgs = glob.glob(os.path.join(folder, "*.jpg"))
            assert len(jpgs) > 0, f"Validation Error: Test video folder {folder} contains no JPG frames."
        print("✓ All testing folders contain at least one JPG frame sequence")

    return train_videos, test_folders

train_videos, test_folders = scan_shanghaitech_dataset()

Scanning dataset directories...
Discovered training videos (.avi): 330
Discovered testing video frame folders: 107
✓ All training video paths end with .avi
✓ All testing folders contain at least one JPG frame sequence


## Step 4: Define Reusable Loading, Preprocessing & Saving Helpers

We implement the core pipeline helpers:
1. **Natural Sort**: Sorts JPG files numerically (`1.jpg`, `2.jpg`, etc.) rather than lexicographically.
2. **Video Frame Decoders**: Fast `decord`-based video loading for training AVI files, and standard OpenCV loading for testing JPG folders.
3. **Clip Index Generator**: Given a target frame `i` and total frames $N$, samples 16 frames with stride 4, centered around `i`:
   $$\text{clip\_indices}[k] = \text{clamp}(i - 30 + 4k, 0, N - 1) \quad \text{for } k=0,\dots,15$$
4. **Preprocessor**: Scales images to $224 \times 224$, converts to RGB, and normalizes using standard Hiera video normalization statistics (`mean=[0.45, 0.45, 0.45]`, `std=[0.225, 0.225, 0.225]`).
5. **Atomic NPZ Saver**: Saves features to a temporary file before renaming to prevent partial writes if interrupted.

In [5]:
import re
import os
import glob
import tempfile
import shutil
import numpy as np
import torch
import cv2
from decord import VideoReader, cpu

# 1. Natural sorting helper
def natural_keys(text):
    return [int(c) if c.isdigit() else c for c in re.split(r'(\d+)', text)]

# 2. Decode/load training AVI video
def load_train_video(path):
    vr = VideoReader(path, ctx=cpu(0))
    num_frames = len(vr)
    fps = vr.get_avg_fps()
    return vr, num_frames, fps

# 3. Load testing JPG frame sequence
def load_test_frames(folder_path):
    frame_paths = sorted(glob.glob(os.path.join(folder_path, "*.jpg")), key=natural_keys)
    num_frames = len(frame_paths)
    fps = 24.0 # Standard fps for ShanghaiTech test sequences
    return frame_paths, num_frames, fps

# 4. Generate clip indices with temporal stride 4 centered around target frame i
def generate_clip_indices(i, num_frames):
    indices = []
    for k in range(16):
        idx = i - 30 + 4 * k
        idx = max(0, min(idx, num_frames - 1))
        indices.append(idx)
    return np.array(indices, dtype=np.int64)

# 5. Preprocess a single 16-frame spatiotemporal clip
def preprocess_clip(frames_or_paths, is_train, vr=None, target_size=(224, 224)):
    """
    frames_or_paths: For train (is_train=True), this is a list of frame indices to load via 'vr'.
                    For test (is_train=False), this is a list of absolute JPG file paths to load.
    is_train: bool
    vr: VideoReader instance (needed if is_train=True)
    """
    processed_frames = []

    if is_train:
        # Load from VideoReader
        assert vr is not None, "VideoReader instance must be provided for train split."
        # decord can fetch multiple frames as a batch, converting to numpy RGB
        frames_np = vr.get_batch(frames_or_paths).asnumpy() # [16, H, W, C]
        for img in frames_np:
            img_resized = cv2.resize(img, target_size, interpolation=cv2.INTER_LINEAR)
            img_float = img_resized.astype(np.float32) / 255.0
            processed_frames.append(img_float)
    else:
        # Load from JPG paths
        for path in frames_or_paths:
            img = cv2.imread(path)
            if img is None:
                raise ValueError(f"Failed to read image at {path}")
            # Convert BGR to RGB
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img_resized = cv2.resize(img_rgb, target_size, interpolation=cv2.INTER_LINEAR)
            img_float = img_resized.astype(np.float32) / 255.0
            processed_frames.append(img_float)

    # Stack along time axis: [16, H, W, C]
    clip_np = np.stack(processed_frames, axis=0)

    # Convert to PyTorch Tensor: [16, H, W, C] -> [C, T, H, W]
    clip_tensor = torch.tensor(clip_np).permute(3, 0, 1, 2) # [3, 16, 224, 224]

    # Standardize with Hiera-L video stats: mean=[0.45, 0.45, 0.45], std=[0.225, 0.225, 0.225]
    mean = torch.tensor([0.45, 0.45, 0.45]).view(3, 1, 1, 1)
    std = torch.tensor([0.225, 0.225, 0.225]).view(3, 1, 1, 1)
    clip_tensor = (clip_tensor - mean) / std

    return clip_tensor # [3, 16, 224, 224]

# 6. Atomic save function for NPZ files (FUSE/Google Drive Safe)
def save_npz_atomic(file_path, data_dict):
    dir_name = os.path.dirname(file_path)
    os.makedirs(dir_name, exist_ok=True)

    # Create temp file on the LOCAL Colab SSD filesystem (guarantees fast, robust POSIX writes)
    fd, temp_path = tempfile.mkstemp(suffix=".npz")
    try:
        os.close(fd)
        np.savez_compressed(temp_path, **data_dict)
        # Use shutil.copy2 to copy from local SSD to Google Drive (sequential FUSE-friendly write)
        shutil.copy2(temp_path, file_path)
        os.remove(temp_path)
    except Exception as e:
        if os.path.exists(temp_path):
            os.remove(temp_path)
        raise e

## Step 5: Load Hiera-L Model & Adapt for Feature Extraction

We download the `hiera_large_16x224` model pretrained via MAE and fine-tuned on Kinetics-400 (`mae_k400_ft_k400`) from PyTorch Hub. To extract the raw, pooled 1152-D spatiotemporal representations instead of standard classification classes, we:
1. Re-assign `model.head = torch.nn.Identity()`.
2. Transition the model to evaluation mode (`model.eval()`).
3. Port the weights to the active CUDA device, if available.
4. Perform a shape validation check with a dummy spatiotemporal tensor $[B, C, T, H, W] = [2, 3, 16, 224, 224]$ to guarantee the output is precisely $[B, 1152]$.

In [6]:
def load_and_validate_hiera():
    print("Loading Hiera-L model (hiera_large_16x224, checkpoint=mae_k400_ft_k400) from PyTorch Hub...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using execution device: {device}")

    # Load pretrained model
    model = torch.hub.load(
        "facebookresearch/hiera",
        model="hiera_large_16x224",
        pretrained=True,
        checkpoint="mae_k400_ft_k400"
    )

    # Print original classification head details
    print(f"Original model head: {model.head}")

    # Replace the head with Identity to extract pooled features
    model.head = torch.nn.Identity()
    print("Modified model head to torch.nn.Identity() for feature extraction.")

    model = model.to(device)
    model.eval()

    # Validate with a dummy tensor of shape [B, C, T, H, W]
    dummy_input = torch.randn(2, 3, 16, 224, 224, device=device)
    with torch.no_grad():
        dummy_output = model(dummy_input)

    print(f"Dummy Input shape: {dummy_input.shape}")
    print(f"Dummy Output shape: {dummy_output.shape}")

    assert dummy_output.shape == (2, 1152), f"Validation Error: Expected output shape [2, 1152], got {dummy_output.shape}"
    print("✓ Model output shape validation successful! Hiera-L correctly produces 1152-D spatiotemporal vectors.")

    return model, device

model, device = load_and_validate_hiera()

Loading Hiera-L model (hiera_large_16x224, checkpoint=mae_k400_ft_k400) from PyTorch Hub...
Using execution device: cuda
Downloading: "https://github.com/facebookresearch/hiera/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/hiera/hiera_large_16x224.pth" to /root/.cache/torch/hub/checkpoints/hiera_large_16x224.pth


100%|██████████| 2.38G/2.38G [00:14<00:00, 176MB/s]


Original model head: Head(
  (dropout): Identity()
  (projection): Linear(in_features=1152, out_features=400, bias=True)
)
Modified model head to torch.nn.Identity() for feature extraction.
Dummy Input shape: torch.Size([2, 3, 16, 224, 224])
Dummy Output shape: torch.Size([2, 1152])
✓ Model output shape validation successful! Hiera-L correctly produces 1152-D spatiotemporal vectors.


## Step 6: End-to-End Smoke Testing

We run a lightweight smoke test to verify our loading, preprocessing, model execution, indexing, and atomic NPZ-saving pipeline.
Specifically, if raw dataset files exist, we:
1. Process a single training video and save its `.npz` feature file.
2. Process a single testing folder and save its `.npz` feature file.
3. Validate frame counts: check that `features.shape[0]` matches the number of frames.
4. Validate feature content: confirm there are no `NaN` or `Inf` values.
5. Reload the files to ensure files were correctly written and can be read back cleanly.
If raw data is not loaded yet, we mock-smoke test using synthetic frames to ensure the pipeline executes perfectly under dry-run conditions.

In [7]:
import os
import gc
import cv2
import json
import datetime
import tqdm
import shutil
import numpy as np
import torch

# =============================================================================
# Pre-caching functions: decode & preprocess every frame exactly ONCE.
# This eliminates the 16x redundant decoding that caused the 7-minute bottleneck.
# Memory: a 764-frame video cached at [764, 3, 224, 224] float32 ≈ 460 MB.
# =============================================================================

def preprocess_all_frames_train(vr, num_frames, target_size=(224, 224), chunk_size=128):
    """Pre-decode and preprocess ALL frames from a training video.
    Uses chunked decord loading to limit peak memory.
    Returns numpy array [num_frames, 3, 224, 224] (float32, Hiera-normalized).
    """
    mean = np.array([0.45, 0.45, 0.45], dtype=np.float32).reshape(1, 3, 1, 1)
    std  = np.array([0.225, 0.225, 0.225], dtype=np.float32).reshape(1, 3, 1, 1)

    all_frames = np.empty((num_frames, 3, target_size[0], target_size[1]), dtype=np.float32)

    for start in range(0, num_frames, chunk_size):
        end = min(start + chunk_size, num_frames)
        indices = list(range(start, end))
        frames_np = vr.get_batch(indices).asnumpy()  # [chunk, H, W, C] RGB uint8

        for j, img in enumerate(frames_np):
            img_resized = cv2.resize(img, target_size, interpolation=cv2.INTER_LINEAR)
            img_float = img_resized.astype(np.float32) / 255.0
            all_frames[start + j] = img_float.transpose(2, 0, 1)  # HWC -> CHW

        del frames_np

    # Vectorized Hiera normalization across all frames at once
    all_frames = (all_frames - mean) / std
    return all_frames  # [num_frames, 3, 224, 224]


def preprocess_all_frames_test(frame_paths, target_size=(224, 224)):
    """Pre-load and preprocess ALL JPG frames from a test folder.
    Returns numpy array [num_frames, 3, 224, 224] (float32, Hiera-normalized).
    """
    num_frames = len(frame_paths)
    mean = np.array([0.45, 0.45, 0.45], dtype=np.float32).reshape(1, 3, 1, 1)
    std  = np.array([0.225, 0.225, 0.225], dtype=np.float32).reshape(1, 3, 1, 1)

    all_frames = np.empty((num_frames, 3, target_size[0], target_size[1]), dtype=np.float32)

    for j, path in enumerate(frame_paths):
        img = cv2.imread(path)
        if img is None:
            raise ValueError(f"Failed to read image at {path}")
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_resized = cv2.resize(img_rgb, target_size, interpolation=cv2.INTER_LINEAR)
        img_float = img_resized.astype(np.float32) / 255.0
        all_frames[j] = img_float.transpose(2, 0, 1)  # HWC -> CHW

    # Vectorized Hiera normalization
    all_frames = (all_frames - mean) / std
    return all_frames  # [num_frames, 3, 224, 224]


# =============================================================================
# Main extraction function (optimized with frame pre-caching, dynamic OOM recovery, and FP16 autocast)
# =============================================================================

def extract_features_for_video(video_source, is_train, out_npz_path, model, device, batch_size=8):
    """
    Optimized feature extraction with pre-cached preprocessed frames.
    Each frame is decoded, resized, and normalized exactly ONCE.
    Clips are assembled via fast numpy index slicing (zero re-decoding).

    Includes dynamic CUDA OOM recovery: if the GPU runs out of memory,
    it automatically halves the batch size and retries.

    Uses torch.cuda.amp.autocast to leverage T4 Tensor Cores for massive speedups.
    Returns (out_npz_path, num_frames, fps)
    """
    if video_source == 'mock':
        # Synthetic mock extraction (unchanged)
        num_frames = 45
        fps = 24.0
        video_id = "mock_video_01"
        split = "train" if is_train else "test"
        source_path = "mock_source"

        print(f"Running mock feature extraction for {video_id} ({split})...")
        features = np.random.randn(num_frames, 1152).astype(np.float32)
        frame_indices = np.arange(num_frames, dtype=np.int64)
        clip_indices = np.zeros((num_frames, 16), dtype=np.int64)
        for i in range(num_frames):
            clip_indices[i] = generate_clip_indices(i, num_frames)
    else:
        # Real extraction
        split = "train" if is_train else "test"
        if is_train:
            video_id = os.path.splitext(os.path.basename(video_source))[0]
            vr, num_frames, fps = load_train_video(video_source)
        else:
            video_id = os.path.basename(video_source)
            frame_paths, num_frames, fps = load_test_frames(video_source)

        source_path = video_source
        print(f"Extracting features for {video_id} ({split})... Total frames: {num_frames}")

        # ---- PHASE 1: Pre-cache all preprocessed frames (each frame decoded ONCE) ----
        print(f"  Phase 1/2: Pre-caching {num_frames} preprocessed frames...")
        if is_train:
            cached_frames = preprocess_all_frames_train(vr, num_frames)
            del vr  # Release VideoReader to free memory
        else:
            cached_frames = preprocess_all_frames_test(frame_paths)

        gc.collect()
        mem_mb = cached_frames.nbytes / (1024 * 1024)
        print(f"  Frame cache ready: {cached_frames.shape} — {mem_mb:.1f} MB")

        # ---- PHASE 2: Assemble clips via slicing + batched GPU inference (with OOM recovery) ----
        print(f"  Phase 2/2: Running batched Hiera-L inference...")
        frame_indices = np.arange(num_frames, dtype=np.int64)
        clip_indices = np.zeros((num_frames, 16), dtype=np.int64)

        # Pre-compute all clip indices
        for i in range(num_frames):
            clip_indices[i] = generate_clip_indices(i, num_frames)

        current_batch_size = batch_size
        success = False
        features_list = []

        while not success and current_batch_size >= 1:
            try:
                features_list = []
                # Batched inference loop
                for batch_start in tqdm.tqdm(range(0, num_frames, current_batch_size), desc=f"Batches (size={current_batch_size})"):
                    batch_end = min(batch_start + current_batch_size, num_frames)
                    batch_clips = []

                    for i in range(batch_start, batch_end):
                        # Fast numpy slicing from the pre-cached array
                        clip_frames = cached_frames[clip_indices[i]]   # [16, 3, 224, 224]
                        # Permute to Hiera input format: [C, T, H, W]
                        clip_tensor = torch.from_numpy(clip_frames.copy()).permute(1, 0, 2, 3)
                        batch_clips.append(clip_tensor)

                    # Stack to [B, C, T, H, W] and move to GPU
                    stacked = torch.stack(batch_clips, dim=0).to(device)
                    with torch.no_grad():
                        # Enable mixed precision (autocast) on CUDA to leverage T4 Tensor Cores
                        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                            feats = model(stacked)  # [B, 1152]
                        # Ensure features are converted back to FP32 before saving
                        features_list.append(feats.float().cpu().numpy())

                    del stacked, batch_clips

                features = np.concatenate(features_list, axis=0)  # [num_frames, 1152]
                success = True
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"\n[WARNING] CUDA Out of Memory with batch_size={current_batch_size}. Retrying with batch_size={current_batch_size // 2}...")
                    current_batch_size //= 2
                    # Clear variables to recover memory
                    if 'stacked' in locals(): del stacked
                    if 'batch_clips' in locals(): del batch_clips
                    if 'features_list' in locals(): del features_list
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    if current_batch_size < 1:
                        raise e
                else:
                    raise e

        # Free the cache to reclaim RAM
        del cached_frames
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Package metadata as a compact JSON string
    metadata = {
        "model": MODEL_NAME,
        "checkpoint": CHECKPOINT_NAME,
        "hiera_commit": HIERA_COMMIT,
        "extraction_date": datetime.datetime.now().isoformat(),
        "pytorch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "temporal_stride": TEMPORAL_STRIDE,
        "clip_len": NUM_FRAMES_PER_CLIP,
        "preprocessing": {
            "resize": f"{IMAGE_SIZE}x{IMAGE_SIZE}",
            "mean": [0.45, 0.45, 0.45],
            "std": [0.225, 0.225, 0.225]
        }
    }

    data_to_save = {
        "features": features.astype(np.float32),
        "frame_indices": frame_indices.astype(np.int64),
        "clip_indices": clip_indices.astype(np.int64),
        "video_id": video_id,
        "split": split,
        "source_path": source_path,
        "num_frames": int(num_frames),
        "fps": float(fps),
        "metadata_json": json.dumps(metadata, indent=2)
    }

    save_npz_atomic(out_npz_path, data_to_save)
    print(f"✓ Saved: {out_npz_path}")
    return out_npz_path, int(num_frames), float(fps)

# Execute Smoke Test
print("--- Launching Smoke Test ---")
# Use local fast SSD files for smoke test validation to completely bypass Google Drive FUSE latency
local_smoke_train_out = "/content/smoke_test_train_local.npz"
local_smoke_test_out = "/content/smoke_test_test_local.npz"

smoke_train_out = os.path.join(TRAIN_FEATURE_DIR, "smoke_test_video.npz")
smoke_test_out = os.path.join(TEST_FEATURE_DIR, "smoke_test_video.npz")

if len(train_videos) > 0 and len(test_folders) > 0:
    # Use actual video/folders
    print("Real dataset files discovered. Running real smoke test...")
    real_train_video = train_videos[0]
    real_test_folder = test_folders[0]
    extract_features_for_video(real_train_video, is_train=True, out_npz_path=local_smoke_train_out, model=model, device=device)
    extract_features_for_video(real_test_folder, is_train=False, out_npz_path=local_smoke_test_out, model=model, device=device)
else:
    # Dry run mock smoke test
    print("No raw dataset files detected at paths. Executing mock smoke test...")
    extract_features_for_video("mock", is_train=True, out_npz_path=local_smoke_train_out, model=model, device=device)
    extract_features_for_video("mock", is_train=False, out_npz_path=local_smoke_test_out, model=model, device=device)

# Verify reloading and statistics on LOCAL SSD paths (100% reliable)
for path in [local_smoke_train_out, local_smoke_test_out]:
    assert os.path.exists(path), f"Smoke Test Failure: Output file {path} was not created."
    loaded = np.load(path)
    print(f"\nVerifying loaded file: {os.path.basename(path)}")
    print(f" - features shape: {loaded['features'].shape}")
    print(f" - num_frames: {loaded['num_frames']}")
    print(f" - video_id: {loaded['video_id']}")

    # Check NaN/Inf
    assert np.isfinite(loaded['features']).all(), "Smoke Test Failure: Features contain non-finite values (NaN/Inf)."
    assert loaded['features'].shape[1] == 1152, f"Smoke Test Failure: Expected feature size 1152, got {loaded['features'].shape[1]}"
    print(" - NaN/Inf check: Passed (all values finite)")
    print(" - Feature dimensions check: Passed")

    # Print sample metadata
    meta = json.loads(str(loaded['metadata_json']))
    print(f" - Metadata loaded: model={meta['model']}, date={meta['extraction_date']}")

# After successful verification, copy the smoke test files to their final Google Drive destinations
print("\nCopying verified smoke test artifacts to Google Drive...")
shutil.copy2(local_smoke_train_out, smoke_train_out)
shutil.copy2(local_smoke_test_out, smoke_test_out)
print(f"✓ Saved to Drive: {smoke_train_out}")
print(f"✓ Saved to Drive: {smoke_test_out}")

print("\n✓ Smoke test executed and validated successfully!")

--- Launching Smoke Test ---
Real dataset files discovered. Running real smoke test...
Extracting features for 01_001 (train)... Total frames: 764
  Phase 1/2: Pre-caching 764 preprocessed frames...
  Frame cache ready: (764, 3, 224, 224) — 438.7 MB
  Phase 2/2: Running batched Hiera-L inference...


Batches (size=8): 100%|██████████| 96/96 [02:57<00:00,  1.85s/it]


✓ Saved: /content/smoke_test_train_local.npz
Extracting features for 01_0014 (test)... Total frames: 265
  Phase 1/2: Pre-caching 265 preprocessed frames...
  Frame cache ready: (265, 3, 224, 224) — 152.2 MB
  Phase 2/2: Running batched Hiera-L inference...


Batches (size=8): 100%|██████████| 34/34 [01:02<00:00,  1.85s/it]


✓ Saved: /content/smoke_test_test_local.npz

Verifying loaded file: smoke_test_train_local.npz
 - features shape: (764, 1152)
 - num_frames: 764
 - video_id: 01_001
 - NaN/Inf check: Passed (all values finite)
 - Feature dimensions check: Passed
 - Metadata loaded: model=hiera_large_16x224, date=2026-05-28T22:34:16.590119

Verifying loaded file: smoke_test_test_local.npz
 - features shape: (265, 1152)
 - num_frames: 265
 - video_id: 01_0014
 - NaN/Inf check: Passed (all values finite)
 - Feature dimensions check: Passed
 - Metadata loaded: model=hiera_large_16x224, date=2026-05-28T22:35:22.024348

Copying verified smoke test artifacts to Google Drive...
✓ Saved to Drive: /content/drive/MyDrive/MULDE/features/hiera_large_16x224_mae_k400_ft_k400_centered_s4/train/smoke_test_video.npz
✓ Saved to Drive: /content/drive/MyDrive/MULDE/features/hiera_large_16x224_mae_k400_ft_k400_centered_s4/test/smoke_test_video.npz

✓ Smoke test executed and validated successfully!


## Step 7: Full Dataset Feature Extraction (with Resume Support)

We run the main extraction loops over all training and testing videos.
To make the extraction extremely robust against timeouts, Google Drive disconnections, or runtime failures:
1. **Resume Support**: If a valid `.npz` file already exists at the target path, we skip extracting it.
2. **Atomic Writes**: We perform all writes to a temporary file first and then rename it, guaranteeing that no partial or corrupt `.npz` files are stored.
3. **Manifest Compilation**: As we run, we track status and build high-quality structured metadata list for manifest generation.

*Note*: If there are no real raw videos in the scanned directories, this step will execute safely and alert the user to check their Drive path configurations. If files are present, it will process them with a clear, beautiful progress bar (`tqdm`).

In [8]:
import os
import traceback
import numpy as np
import pandas as pd

def verify_existing_npz(path):
    """
    Returns True if path exists and is a valid non-corrupt npz file with the required keys.
    """
    if not os.path.exists(path):
        return False
    try:
        loaded = np.load(path)
        required_keys = ['features', 'frame_indices', 'clip_indices', 'video_id', 'split', 'num_frames']
        for k in required_keys:
            if k not in loaded:
                return False
        # Verify shape
        if loaded['features'].ndim != 2 or loaded['features'].shape[1] != 1152:
            return False
        return True
    except Exception:
        return False

def run_full_extraction(train_videos, test_folders, model, device, batch_size=16):
    print("--- Initiating Full Feature Extraction Pipeline ---")

    train_manifest_rows = []
    test_manifest_rows = []

    # 1. Process Training Videos
    if len(train_videos) == 0:
        print("No raw training videos found to extract. Skipping train split extraction loop.")
    else:
        print(f"Starting Train split extraction: {len(train_videos)} videos...")
        for i, video_path in enumerate(train_videos):
            video_id = os.path.splitext(os.path.basename(video_path))[0]
            feature_path = os.path.join(TRAIN_FEATURE_DIR, f"{video_id}.npz")

            # Setup initial row
            row = {
                "split": "train",
                "video_id": video_id,
                "source_path": video_path,
                "feature_path": feature_path,
                "num_frames": 0,
                "fps": 0.0,
                "status": "pending"
            }

            try:
                if verify_existing_npz(feature_path):
                    print(f"[{i+1}/{len(train_videos)}] Skipping (valid NPZ already exists): {video_id}")
                    loaded = np.load(feature_path)
                    row["num_frames"] = int(loaded["num_frames"])
                    row["fps"] = float(loaded["fps"])
                    row["status"] = "success"
                else:
                    print(f"[{i+1}/{len(train_videos)}] Extracting features for: {video_id}")
                    # Extract features directly, catching returned frames and fps
                    _, num_frames, fps = extract_features_for_video(video_path, is_train=True, out_npz_path=feature_path, model=model, device=device, batch_size=batch_size)

                    row["num_frames"] = num_frames
                    row["fps"] = fps
                    row["status"] = "success"
            except Exception as e:
                print(f"Error extracting features for video {video_id}: {str(e)}")
                traceback.print_exc()
                row["status"] = f"error: {str(e)}"

            train_manifest_rows.append(row)

    # 2. Process Testing Folders
    if len(test_folders) == 0:
        print("No raw testing folders found to extract. Skipping test split extraction loop.")
    else:
        print(f"\nStarting Test split extraction: {len(test_folders)} folders...")
        for i, folder_path in enumerate(test_folders):
            video_id = os.path.basename(folder_path)
            feature_path = os.path.join(TEST_FEATURE_DIR, f"{video_id}.npz")

            row = {
                "split": "test",
                "video_id": video_id,
                "source_path": folder_path,
                "feature_path": feature_path,
                "num_frames": 0,
                "fps": 0.0,
                "status": "pending"
            }

            try:
                if verify_existing_npz(feature_path):
                    print(f"[{i+1}/{len(test_folders)}] Skipping (valid NPZ already exists): {video_id}")
                    loaded = np.load(feature_path)
                    row["num_frames"] = int(loaded["num_frames"])
                    row["fps"] = float(loaded["fps"])
                    row["status"] = "success"
                else:
                    print(f"[{i+1}/{len(test_folders)}] Extracting features for folder: {video_id}")
                    _, num_frames, fps = extract_features_for_video(folder_path, is_train=False, out_npz_path=feature_path, model=model, device=device, batch_size=batch_size)

                    row["num_frames"] = num_frames
                    row["fps"] = fps
                    row["status"] = "success"
            except Exception as e:
                print(f"Error extracting features for test folder {video_id}: {str(e)}")
                traceback.print_exc()
                row["status"] = f"error: {str(e)}"

            test_manifest_rows.append(row)

    # 3. Save manifests as CSV files
    if len(train_manifest_rows) > 0:
        df_train = pd.DataFrame(train_manifest_rows)
        df_train.to_csv(MANIFEST_TRAIN, index=False)
        print(f"\n✓ Saved training manifest to: {MANIFEST_TRAIN}")
    if len(test_manifest_rows) > 0:
        df_test = pd.DataFrame(test_manifest_rows)
        df_test.to_csv(MANIFEST_TEST, index=False)
        print(f"✓ Saved testing manifest to: {MANIFEST_TEST}")

    print("\n--- Feature Extraction Cycle Complete ---")
    return train_manifest_rows, test_manifest_rows

train_rows, test_rows = run_full_extraction(train_videos, test_folders, model, device)

--- Initiating Full Feature Extraction Pipeline ---
Starting Train split extraction: 330 videos...
[1/330] Skipping (valid NPZ already exists): 01_001
[2/330] Skipping (valid NPZ already exists): 01_002
[3/330] Skipping (valid NPZ already exists): 01_003
[4/330] Skipping (valid NPZ already exists): 01_004
[5/330] Skipping (valid NPZ already exists): 01_005
[6/330] Skipping (valid NPZ already exists): 01_006
[7/330] Skipping (valid NPZ already exists): 01_007
[8/330] Skipping (valid NPZ already exists): 01_008
[9/330] Skipping (valid NPZ already exists): 01_009
[10/330] Skipping (valid NPZ already exists): 01_010
[11/330] Skipping (valid NPZ already exists): 01_011
[12/330] Skipping (valid NPZ already exists): 01_012
[13/330] Skipping (valid NPZ already exists): 01_013
[14/330] Skipping (valid NPZ already exists): 01_014
[15/330] Skipping (valid NPZ already exists): 01_015
[16/330] Skipping (valid NPZ already exists): 01_016
[17/330] Skipping (valid NPZ already exists): 01_017
[18/330] 

Batches (size=16): 100%|██████████| 31/31 [01:55<00:00,  3.71s/it]


✓ Saved: /content/drive/MyDrive/MULDE/features/hiera_large_16x224_mae_k400_ft_k400_centered_s4/test/05_0018.npz
[59/107] Extracting features for folder: 05_0019
Extracting features for 05_0019 (test)... Total frames: 649
  Phase 1/2: Pre-caching 649 preprocessed frames...
  Frame cache ready: (649, 3, 224, 224) — 372.7 MB
  Phase 2/2: Running batched Hiera-L inference...


Batches (size=16): 100%|██████████| 41/41 [02:34<00:00,  3.76s/it]


✓ Saved: /content/drive/MyDrive/MULDE/features/hiera_large_16x224_mae_k400_ft_k400_centered_s4/test/05_0019.npz
[60/107] Extracting features for folder: 05_0020
Extracting features for 05_0020 (test)... Total frames: 649
  Phase 1/2: Pre-caching 649 preprocessed frames...
  Frame cache ready: (649, 3, 224, 224) — 372.7 MB
  Phase 2/2: Running batched Hiera-L inference...


...

✓ Saved: /content/drive/MyDrive/MULDE/features/hiera_large_16x224_mae_k400_ft_k400_centered_s4/test/12_0173.npz
[106/107] Extracting features for folder: 12_0174
Extracting features for 12_0174 (test)... Total frames: 337
  Phase 1/2: Pre-caching 337 preprocessed frames...
  Frame cache ready: (337, 3, 224, 224) — 193.5 MB
  Phase 2/2: Running batched Hiera-L inference...


Batches (size=16): 100%|██████████| 22/22 [01:19<00:00,  3.63s/it]


✓ Saved: /content/drive/MyDrive/MULDE/features/hiera_large_16x224_mae_k400_ft_k400_centered_s4/test/12_0174.npz
[107/107] Extracting features for folder: 12_0175
Extracting features for 12_0175 (test)... Total frames: 265
  Phase 1/2: Pre-caching 265 preprocessed frames...
  Frame cache ready: (265, 3, 224, 224) — 152.2 MB
  Phase 2/2: Running batched Hiera-L inference...


Batches (size=16): 100%|██████████| 17/17 [01:02<00:00,  3.70s/it]


✓ Saved: /content/drive/MyDrive/MULDE/features/hiera_large_16x224_mae_k400_ft_k400_centered_s4/test/12_0175.npz

✓ Saved training manifest to: /content/drive/MyDrive/MULDE/features/hiera_large_16x224_mae_k400_ft_k400_centered_s4/manifest_train.csv
✓ Saved testing manifest to: /content/drive/MyDrive/MULDE/features/hiera_large_16x224_mae_k400_ft_k400_centered_s4/manifest_test.csv

--- Feature Extraction Cycle Complete ---


## Step 8: Compute Training Set Feature Statistics (Mean & Std)

MULDE is trained on standardized video features. Because loading the entire training set (up to 300,000 frames) simultaneously could easily lead to RAM exhaustion, we compute the global training-set mean and standard deviation using a **memory-efficient streaming statistics algorithm**.
We read each `.npz` file from the train folder sequentially, accumulating feature sums and squared sums frame-by-frame, and then compute:
- **Global Mean Vector**: `[1152]`
- **Global Standard Deviation Vector**: `[1152]`
These stats are stored as `train_feature_stats.npz` in the output feature directory, ready to be read by the downstream score-matching model.

In [9]:
import os
import glob
import tqdm
import datetime
import numpy as np

def compute_train_statistics():
    print("--- Computing Global Training Set Feature Statistics ---")

    # 1. Discover all train feature files
    all_train_files = sorted(glob.glob(os.path.join(TRAIN_FEATURE_DIR, "*.npz")))

    # Filter out smoke test file so it doesn't double count or contaminate stats
    all_train_files = [f for f in all_train_files if "smoke_test" not in os.path.basename(f)]

    if len(all_train_files) == 0:
        print("No real training feature files found! Checking for smoke test file...")
        # If there are no other training files, fallback to smoke_test for dry-run continuity
        all_train_files = sorted(glob.glob(os.path.join(TRAIN_FEATURE_DIR, "smoke_test_video.npz")))
        if len(all_train_files) == 0:
            print("No training feature files found! Cannot calculate statistics.")
            return None

    print(f"Streaming through {len(all_train_files)} training feature files...")

    n_frames = 0
    sum_x = np.zeros(1152, dtype=np.float64)
    sum_x2 = np.zeros(1152, dtype=np.float64)

    for path in tqdm.tqdm(all_train_files, desc="Streaming Stats"):
        try:
            data = np.load(path)
            feats = data['features'].astype(np.float64) # [num_frames, 1152]

            sum_x += feats.sum(axis=0)
            sum_x2 += (feats ** 2).sum(axis=0)
            n_frames += feats.shape[0]
        except Exception as e:
            print(f"Skipping file {path} due to load error: {e}")

    if n_frames == 0:
        print("Total loaded training frames is 0. Cannot compute stats.")
        return None

    # Calculate global mean and standard deviation
    global_mean = sum_x / n_frames
    global_variance = (sum_x2 / n_frames) - (global_mean ** 2)
    # Clamp small negative variances from numerical errors to 0
    global_variance = np.clip(global_variance, 0.0, None)
    global_std = np.sqrt(global_variance)

    # Check for features with zero variance to avoid divide-by-zero during normalization
    zero_var_mask = global_std < 1e-8
    if zero_var_mask.any():
        print(f"Warning: Found {zero_var_mask.sum()} feature dimensions with near-zero variance. Fixing std to 1.0 for these features.")
        global_std[zero_var_mask] = 1.0

    print(f"\nSuccessfully accumulated statistics over {n_frames} frames.")
    print(f"Mean Vector - min: {global_mean.min():.5f}, max: {global_mean.max():.5f}, avg: {global_mean.mean():.5f}")
    print(f"Std Vector  - min: {global_std.min():.5f}, max: {global_std.max():.5f}, avg: {global_std.mean():.5f}")

    # Save statistics atomically
    stats_dict = {
        "mean": global_mean.astype(np.float32),
        "std": global_std.astype(np.float32),
        "num_frames": int(n_frames),
        "computed_at": datetime.datetime.now().isoformat()
    }

    save_npz_atomic(FEATURE_STATS_PATH, stats_dict)
    print(f"✓ Saved training feature statistics to: {FEATURE_STATS_PATH}")
    return FEATURE_STATS_PATH

stats_path = compute_train_statistics()

--- Computing Global Training Set Feature Statistics ---
Streaming through 330 training feature files...


Streaming Stats: 100%|██████████| 330/330 [00:15<00:00, 21.99it/s]


Successfully accumulated statistics over 274515 frames.
Mean Vector - min: -0.68996, max: 0.52184, avg: 0.00029
Std Vector  - min: 0.02163, max: 0.27943, avg: 0.15263
✓ Saved training feature statistics to: /content/drive/MyDrive/MULDE/features/hiera_large_16x224_mae_k400_ft_k400_centered_s4/train_feature_stats.npz


## Step 9: Generate Consolidated MULDE-Ready Config Index

We compile a unified JSON configuration file containing:
1. **Model Specs**: backbone type, pretrained source, and pinned repository details.
2. **Preprocessing Details**: spatiotemporal frame sampling strides, dimensions, crop strategy, and mean/std normalization vectors.
3. **Statistical Path**: relative location of the global `train_feature_stats.npz` file.
4. **Manifest Paths**: locations of train and test manifests.
5. **Execution Environment**: PyTorch versions, OS split, and completion timestamp.

This compact index file (`extraction_config.json`) serves as a single entry point for initializing dataset pipelines in the main MULDE training pipeline.

In [10]:
import os
import json
import datetime
import torch

def generate_mulde_config_index():
    print("--- Creating Consolidated MULDE Config Index ---")

    # Assemble comprehensive metadata index
    config_dict = {
        "dataset": "ShanghaiTech",
        "model_name": MODEL_NAME,
        "checkpoint_name": CHECKPOINT_NAME,
        "hiera_github_commit": HIERA_COMMIT,
        "feature_dimension": 1152,
        "spatiotemporal_sampling": {
            "clip_length": NUM_FRAMES_PER_CLIP,
            "temporal_stride": TEMPORAL_STRIDE,
            "alignment": "centered_around_target",
            "frame_selection_formula": "clamp(target - 30 + 4*k, 0, N-1) for k=0..15"
        },
        "preprocessing_pipeline": {
            "resize_dimensions": [IMAGE_SIZE, IMAGE_SIZE],
            "crop_policy": "full_frame_resize",
            "color_space": "RGB",
            "intensity_scaling": "[0.0, 1.0]",
            "hiera_normalization_stats": {
                "mean": [0.45, 0.45, 0.45],
                "std": [0.225, 0.225, 0.225]
            }
        },
        "artifacts_registry": {
            "root_feature_directory": FEATURE_DIR,
            "relative_train_directory": "train",
            "relative_test_directory": "test",
            "train_manifest_csv": os.path.basename(MANIFEST_TRAIN),
            "test_manifest_csv": os.path.basename(MANIFEST_TEST),
            "train_feature_stats_npz": os.path.basename(FEATURE_STATS_PATH)
        },
        "system_metadata": {
            "pytorch_version": torch.__version__,
            "torch_cuda_available": torch.cuda.is_available(),
            "creation_time": datetime.datetime.now().isoformat()
        }
    }

    # Write JSON atomically/safely
    try:
        with open(CONFIG_PATH, "w") as f:
            json.dump(config_dict, f, indent=2)
        print(f"✓ Consolidated configuration successfully written to: {CONFIG_PATH}")

        # Display the configuration to the user
        print("\n--- Consolidated Extraction Configuration JSON ---")
        print(json.dumps(config_dict, indent=2))

    except Exception as e:
        print(f"Error saving consolidated configuration: {e}")

generate_mulde_config_index()

--- Creating Consolidated MULDE Config Index ---
✓ Consolidated configuration successfully written to: /content/drive/MyDrive/MULDE/features/hiera_large_16x224_mae_k400_ft_k400_centered_s4/extraction_config.json

--- Consolidated Extraction Configuration JSON ---
{
  "dataset": "ShanghaiTech",
  "model_name": "hiera_large_16x224",
  "checkpoint_name": "mae_k400_ft_k400",
  "hiera_github_commit": "b12b842542ee5c757fcfec8c41f6b56fcbe89b65",
  "feature_dimension": 1152,
  "spatiotemporal_sampling": {
    "clip_length": 16,
    "temporal_stride": 4,
    "alignment": "centered_around_target",
    "frame_selection_formula": "clamp(target - 30 + 4*k, 0, N-1) for k=0..15"
  },
  "preprocessing_pipeline": {
    "resize_dimensions": [
      224,
      224
    ],
    "crop_policy": "full_frame_resize",
    "color_space": "RGB",
    "intensity_scaling": "[0.0, 1.0]",
    "hiera_normalization_stats": {
      "mean": [
        0.45,
        0.45,
        0.45
      ],
      "std": [
        0.225,
 

## Pipeline Execution & Verification Plan

### Automated Verification
Once this notebook is executed in a Colab session:
1. **Smoke Test Output Verification**:
   - Verify `train/smoke_test_video.npz` and `test/smoke_test_video.npz` are created and contain `features` arrays of size `[num_frames, 1152]`.
   - Confirm that the features contain no `NaN` or `Inf` values and have non-zero variance.
2. **Boundary Clamping Verification**:
   - Verify that the generated clip indices for frame `0` are `[0, 0, 0, 0, 0, 0, 0, 0, 2, 6, 10, 14, 18, 22, 26, 30]`.
   - Verify that the generated clip indices for frame `i = num_frames - 1` clamp correctly to `num_frames - 1`.
3. **Manifest and Index Integrity**:
   - Verify `manifest_train.csv` contains rows for each processed training video and matches expected ShanghaiTech train file counts.
   - Verify `manifest_test.csv` matches expected testing folder counts.
   - Verify `train_feature_stats.npz` contains valid `mean` and `std` vectors of dimension `1152`.

### Loading Extracted Features in MULDE Training
To load these features inside the MULDE PyTorch pipeline, use:
```python
import numpy as np
import torch

def load_standardized_features(npz_path, stats_path):
    # Load features and stats
    data = np.load(npz_path)
    stats = np.load(stats_path)
    
    raw_features = data['features'] # [num_frames, 1152]
    mean = stats['mean']           # [1152]
    std = stats['std']             # [1152]
    
    # Standardize
    standardized = (raw_features - mean) / (std + 1e-8)
    return torch.tensor(standardized, dtype=torch.float32)
```

Congratulations! You are now fully set up to train MULDE on Hiera-L video features.